# Graph RAG 实战：极简版

本 notebook 展示 Graph RAG 的核心思想：
1. **从文档中抽取实体和关系**，构建知识图谱
2. **利用图谱增强检索**，扩展查询或过滤结果

对比：Naive RAG 只做向量相似度检索，Graph RAG 额外利用结构化的实体关系。

In [12]:
# 环境准备
from __future__ import annotations
import os, json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
import networkx as nx

PROJECT_ROOT = Path('/Users/mengbai/Documents/AI-training/RAG_project')
env_path = PROJECT_ROOT / '.env'
if not env_path.exists():
    env_path = PROJECT_ROOT.parent / '.env'
load_dotenv(env_path, override=True)

api_key = os.getenv('OPENAI_API_KEY')
base_url = os.getenv('OPENAI_BASE_URL', 'https://api.siliconflow.cn/v1')

emb = OpenAIEmbeddings(model='Qwen/Qwen3-Embedding-8B', api_key=api_key, base_url=base_url)
vs = Chroma(collection_name='autel_annual_report_2024', embedding_function=emb, persist_directory=str(PROJECT_ROOT / 'data/chroma'))
llm = ChatOpenAI(model='deepseek-ai/DeepSeek-V3.2', temperature=0, api_key=api_key, base_url=base_url)

# 加载文档
all_data = vs.get(include=['documents', 'metadatas'])
all_docs = [Document(page_content=d, metadata=m) for d, m in zip(all_data['documents'], all_data['metadatas'])]
print(f"加载了 {len(all_docs)} 个文档片段")

加载了 931 个文档片段


## Step 1: 定义查询

我们要回答的问题是：**道通科技的 AI 战略是什么？**

In [13]:
TARGET_SECTION_TITLE = "一、经营情况讨论与分析"
QUERY = "这一节里，道通有哪些产品？"
print(f"Target section title: {TARGET_SECTION_TITLE}")
print(f"Query: {QUERY}")

Target section title: 一、经营情况讨论与分析
Query: 这一节里，道通有哪些产品？


## Step 2: Naive RAG（基线）

直接用向量检索获取 top-k 文档。

In [14]:
# Naive RAG: 纯向量检索
naive_docs = vs.similarity_search(QUERY, k=5)
print("=" * 60)
print("Naive RAG 结果")
print("=" * 60)
for i, d in enumerate(naive_docs, 1):
    print(f"\n[{i}] {d.page_content[:500]}...")

Naive RAG 结果

[1] 房租及管理费主要核算为研发活动而发生的物业管理及水电费。...

[2] $\surd$ 适用 □不适用  
本企业重要的合营或联营企业详见附注十之说明。...

[3] √适用 □不适用  
本公司不存在导致对报告期末起12个月内的持续经营能力产生重大疑虑的事项或情况。...

[4] 一、本公司董事会、监事会及董事、监事、高级管理人员保证年度报告内容的真实性、准确性、完整性，不存在虚假记载、误导性陈述或重大遗漏，并承担个别和连带的法律责任。...

[5] $\surd$ 适用 □不适用  
公司以内部组织结构、管理要求、内部报告制度等为依据确定报告分部，并以地区分部为基础确定报告分部，分别对中国境内、北美地区、欧洲地区、其他地区等的经营业绩进行考核。...


## Step 3: 用 LLM 做实体抽取

从文档中抽取关键实体（公司、产品、技术、人物等）。这是 Graph RAG 的第一步。

In [15]:
# LLM 实体抽取 prompt
ENTITY_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """你是一个信息抽取专家。从给定文本中抽取关键实体。
    输出格式：JSON 数组，每个元素是一个实体，格式如下：
    {{"type": "实体类型", "name": "实体名称", "description": "一句话描述"}}
    实体类型包括：公司、产品、技术、人物、业务、地点、日期、年份。
    只输出 JSON，不要其他内容。"""),
    ("human", "{text}")
])

def extract_entities(text, max_len=800, verbose=False):
    """从文本中抽取实体。对 LLM 返回做鲁棒 JSON 解析。"""
    text = text[:max_len]
    try:
        result = (ENTITY_PROMPT | llm).invoke({"text": text})
        raw = result.content.strip()
        
        # 兼容 ```json ... ``` 包裹
        if raw.startswith("```"):
            raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        
        # 截取第一个 JSON 数组
        start = raw.find("[")
        end = raw.rfind("]")
        if start != -1 and end != -1 and end > start:
            raw = raw[start:end+1]
        
        data = json.loads(raw)
        return data if isinstance(data, list) else []
    except Exception as e:
        if verbose:
            print("实体抽取失败:", e)
            try:
                print("原始返回:", result.content[:500])
            except Exception:
                pass
        return []

# 对更像正文的文档做实体抽取演示（避开表格/票据噪声）
print("实体抽取示例：")
sample_text = all_docs[10].page_content[:500]
print("sample:", sample_text[:120].replace("\n", " "))
entities = extract_entities(sample_text, verbose=True)
print(f"抽取到 {len(entities)} 个实体，前 5 个：{entities[:5]}")

实体抽取示例：
sample: 各位投资者，AI浪潮席卷全球，产业格局正重塑。当此之时，道通科技不做追风者，而争做时代浪潮中的掌舵人。   我们深知，AI 真正的价值，绝非技术炫耀，而在于踏踏实实地为行业解决具体问题——让每辆汽车的诊断更精准，让每一度绿色能源利用更高效，
抽取到 10 个实体，前 5 个：[{'type': '公司', 'name': '道通科技', 'description': '一家从深圳出发走向世界的科技公司，专注于汽车诊断、绿色能源、低空经济和具身智能领域。'}, {'type': '地点', 'name': '深圳', 'description': '道通科技的出发地，也是其全球业务的起点。'}, {'type': '业务', 'name': '绿色能源', 'description': '道通科技未来十年将重点发展的万亿级产业之一，旨在提高能源利用效率。'}, {'type': '业务', 'name': '低空经济', 'description': '道通科技未来十年将重点发展的万亿级产业之一，涉及空中交通和相关服务。'}, {'type': '业务', 'name': '具身智能', 'description': '道通科技未来十年将重点发展的万亿级产业之一，专注于机器人自主协同作业技术。'}]


## Step 4: 构建知识图谱

将抽取的实体存入 NetworkX 图结构，存储实体之间的关系。

In [16]:
# 构建局部知识图谱：以“管理层讨论与分析”正文起点为中心，取前后 20 个 chunk 窗口
G = nx.MultiDiGraph()

# 先按 chunk_in_doc 排序，便于做局部窗口
ordered_docs = sorted(
    all_docs,
    key=lambda d: (
        d.metadata.get("chunk_in_doc") is None,
        d.metadata.get("chunk_in_doc") if d.metadata.get("chunk_in_doc") is not None else 10**9,
    )
)

# 锚定真正正文起点，而不是“详见第三节...”这类引用文本
anchor_candidates = [
    d for d in ordered_docs
    if d.metadata.get("section_title") == TARGET_SECTION_TITLE
]
assert anchor_candidates, f"没找到 section_title={TARGET_SECTION_TITLE!r} 的正文 chunk"

anchor_doc = min(anchor_candidates, key=lambda d: d.metadata.get("chunk_in_doc", 10**9))
anchor_idx = anchor_doc.metadata.get("chunk_in_doc")
start_idx = max(0, anchor_idx - 20)
end_idx = anchor_idx + 20

focused_docs = [
    d for d in ordered_docs
    if d.metadata.get("chunk_in_doc") is not None and start_idx <= d.metadata.get("chunk_in_doc") <= end_idx
]

print(f"anchor section_title: {TARGET_SECTION_TITLE}")
print(f"anchor chunk_in_doc: {anchor_idx}")
print(f"window: [{start_idx}, {end_idx}]")
print(f"focused docs: {len(focused_docs)}")
print(f"anchor text: {anchor_doc.page_content[:200].replace(chr(10), ' ')}")

# 跳过明显纯噪声/纯表格片段，优先抽正文实体
candidate_docs = [
    d for d in focused_docs
    if len(d.page_content.strip()) > 60
    and not d.page_content.strip().startswith("|")
    and "银行承兑汇票" not in d.page_content[:120]
]

print("正在构建局部知识图谱...")
processed = 0
for doc in candidate_docs:
    entities = extract_entities(doc.page_content)
    if not entities:
        continue
    processed += 1

    names = []
    for e in entities:
        if not isinstance(e, dict) or not e.get("name"):
            continue
        G.add_node(
            e["name"],
            type=e.get("type", "其他"),
            description=e.get("description", "")
        )
        names.append(e["name"])

    for a in range(len(names)):
        for b in range(a + 1, len(names)):
            G.add_edge(names[a], names[b], relation="co_occurrence", chunk_in_doc=doc.metadata.get("chunk_in_doc"))

print(f"局部知识图谱构建完成：{G.number_of_nodes()} 个实体，{G.number_of_edges()} 个关系")
print("部分实体示例：")
for node in list(G.nodes())[:12]:
    print(f"  - {node} ({G.nodes[node].get('type', '未知')})")

anchor section_title: 一、经营情况讨论与分析
anchor chunk_in_doc: 44
window: [24, 64]
focused docs: 41
anchor text: 长期以来，公司以 AI 为核心驱动力，紧密围绕“智能化”战略布局业务生态，不断推动AI 技术与业务的深度融合。   2024年，道通科技“全面 AI”，加速推动AI技术与业务场景和组织变革深度融合，进一步巩固数字维修的全球领导者地位，致力于成为智慧能源领域的全球领军企业以及空地一体集群智慧解决方案的全球领导者，努力成为 AI 行业大模型商业化应用的龙头企业。   2024 年，在数字维修业务领域，
正在构建局部知识图谱...
局部知识图谱构建完成：252 个实体，2020 个关系
部分实体示例：
  - 深圳市道通科技股份有限公司 (公司)
  - A股 (产品)
  - 深圳市道合通达投资企业（有限合伙） (公司)
  - 宁波荟顺投资合伙企业（有限合伙） (公司)
  - 深圳市塞防科技有限公司 (公司)
  - 青岛金石灏钠投资有限公司 (公司)
  - 深圳市达晨创恒股权投资企业（有限合伙） (公司)
  - 深圳市达晨创泰股权投资企业（有限合伙） (公司)
  - 深圳市达晨创瑞股权投资企业（有限合伙） (公司)
  - 深圳市达晨创丰股权投资企业（有限合伙） (公司)
  - 深圳市达晨财信创业投资管理有限公司 (公司)
  - 浙江海宁嘉慧投资合伙企业（有限合伙） (公司)


## Step 5: Graph RAG 检索

核心思路：
1. 先用向量召回候选文档
2. **利用知识图谱扩展查询**：根据已有实体，找到相关实体，用扩展后的查询再检索
3. 或者：直接用图中的相关实体来过滤/排序结果

In [18]:
# Graph RAG: 利用图中相邻实体做多路检索

def graph_rag_retrieve(query, k=5, per_entity_k=3):
    """query -> 实体抽取 -> 图邻居 -> 邻居分别召回 -> 融合排序。"""
    query_entities = extract_entities(query)
    print(f"查询中的实体：{query_entities}")

    # 对当前 MultiDiGraph，同时看 successors + predecessors，避免只拿单向邻居
    neighbor_entities = []
    for e in query_entities:
        entity_name = e.get("name")
        if not entity_name or entity_name not in G:
            continue

        neighbors = list(dict.fromkeys(
            list(G.successors(entity_name)) + list(G.predecessors(entity_name))
        ))
        if neighbors:
            picked = neighbors[:3]
            neighbor_entities.extend(picked)
            print(f"  相邻实体 {entity_name} -> {picked}")

    neighbor_entities = list(dict.fromkeys(
        x.strip() for x in neighbor_entities if isinstance(x, str) and x.strip()
    ))
    print(f"最终相邻实体：{neighbor_entities}")

    # 如果图里没有邻居，就退回原始 query 检索
    if not neighbor_entities:
        docs = vs.similarity_search(query, k=k)
        return [{"term": "[fallback] original query", "docs": docs}], docs

    grouped_results = []
    doc_by_key = {}
    fused_scores = {}

    # 每个相邻实体单独召回，再用简单 rank fusion 融合
    for entity in neighbor_entities:
        docs = vs.similarity_search(entity, k=per_entity_k)
        grouped_results.append({"term": entity, "docs": docs})
        print(f"  entity={entity!r} -> {len(docs)} hits")

        for rank, d in enumerate(docs, 1):
            key = (d.metadata.get("chunk_id"), d.page_content[:120])
            doc_by_key[key] = d
            fused_scores[key] = fused_scores.get(key, 0.0) + 1.0 / rank

    merged_docs = [
        doc_by_key[key]
        for key in sorted(fused_scores, key=fused_scores.get, reverse=True)
    ][:k]

    return grouped_results, merged_docs

print("=" * 60)
print("Graph RAG 检索")
print("=" * 60)
graph_groups, graph_docs = graph_rag_retrieve(QUERY, k=5)

for group in graph_groups:
    print(f"--- seed entity: {group['term']} ---")
    for i, d in enumerate(group["docs"], 1):
        print(f"[{i}] {d.page_content[:150]}...")


Graph RAG 检索
查询中的实体：[{'type': '产品', 'name': '道通', 'description': '道通是一家公司，提供多种产品。'}]
  相邻实体 道通 -> ['汽车综合诊断产品', 'TPMS系列产品', 'ADAS标定产品']
最终相邻实体：['汽车综合诊断产品', 'TPMS系列产品', 'ADAS标定产品']
  entity='汽车综合诊断产品' -> 3 hits
  entity='TPMS系列产品' -> 3 hits
  entity='ADAS标定产品' -> 3 hits
--- seed entity: 汽车综合诊断产品 ---
[1] 现代汽车实现了高度电子化，对行业参与者要求必须有长期的技术研发和数据积累以及较强的研发创新能力，这样才能面对不断进化的汽车电子系统时开发出与之相适应的、具备全方位的诊断功能的产品，因而汽车诊断行业具有较强的行业属性和较高的技术壁垒。产品的车型覆盖面、诊断检测结果准确性、功能完整性、使用智能便利性、软...
[2] 各国对新能源以及汽车综合诊断、检测行业相关的产业政策出台将对公司产品销售产生重大影响。如果未来主要国家或地区政府对于新能源、汽车综合诊断、检测行业相关的产业政策发生重大不利变化，可能导致下游客户对公司产品的需求发生波动，进而影响公司的经营业绩。...
[3] √适用 □不适用  
单位：元币种：人民币  
| 合同分类 | 合计 |
| --- | --- |
| | 营业收入 |
| 商品类型 | |
| 汽车诊断产品 | 1,141,183,830.33 |
| TPMS 产品 | 190,832,922.58 |
| ADAS 产品 | 70,36...
--- seed entity: TPMS系列产品 ---
[1] 现代汽车实现了高度电子化，对行业参与者要求必须有长期的技术研发和数据积累以及较强的研发创新能力，这样才能面对不断进化的汽车电子系统时开发出与之相适应的、具备全方位的诊断功能的产品，因而汽车诊断行业具有较强的行业属性和较高的技术壁垒。产品的车型覆盖面、诊断检测结果准确性、功能完整性、使用智能便利性、软...
[2] 单位：元 币种：人民币  
| 主营业务分行业情况 |
| --- |
| 分行业 | 营业收入 | 营业成本 | 毛利率(%

In [19]:

def graph_rag_retrieve(query, k=5, per_entity_k=3):
    """query -> 实体抽取 -> 局部图邻居 -> 邻居分别召回 -> 融合排序。"""
    query_entities = extract_entities(query, verbose=True)
    print(f"查询中的实体：{query_entities}")

    neighbor_entities = []
    for e in query_entities:
        entity_name = e.get("name")
        if not entity_name or entity_name not in G:
            continue

        neighbors = list(dict.fromkeys(
            list(G.successors(entity_name)) + list(G.predecessors(entity_name))
        ))
        if neighbors:
            picked = neighbors[:3]
            neighbor_entities.extend(picked)
            print(f"  相邻实体 {entity_name} -> {picked}")

    neighbor_entities = list(dict.fromkeys(
        x.strip() for x in neighbor_entities if isinstance(x, str) and x.strip()
    ))
    print(f"最终相邻实体：{neighbor_entities}")

    # 如果 query 没抽到实体或图中没有邻居，就退回 focused window 内的 baseline
    if not neighbor_entities:
        fallback_docs = focused_docs[:k]
        return [{"term": "[fallback] focused window", "docs": fallback_docs}], fallback_docs

    grouped_results = []
    doc_by_key = {}
    fused_scores = {}

    for entity in neighbor_entities:
        docs = vs.similarity_search(entity, k=per_entity_k)
        grouped_results.append({"term": entity, "docs": docs})
        print(f"  entity={entity!r} -> {len(docs)} hits")

        for rank, d in enumerate(docs, 1):
            key = (d.metadata.get("chunk_id"), d.page_content[:120])
            doc_by_key[key] = d
            fused_scores[key] = fused_scores.get(key, 0.0) + 1.0 / rank

    merged_docs = [
        doc_by_key[key]
        for key in sorted(fused_scores, key=fused_scores.get, reverse=True)
    ][:k]

    return grouped_results, merged_docs

print("=" * 60)
print("Graph RAG 检索")
print("=" * 60)
graph_groups, graph_docs = graph_rag_retrieve(QUERY, k=5)

for group in graph_groups:
    print()
    print(f"--- seed entity: {group['term']} ---")
    for i, d in enumerate(group["docs"], 1):
        print(f"[{i}] chunk_in_doc={d.metadata.get('chunk_in_doc')} | {d.page_content[:150]}...")

Graph RAG 检索
查询中的实体：[{'type': '产品', 'name': '道通', 'description': '道通是一家公司，提供多种产品。'}]
  相邻实体 道通 -> ['汽车综合诊断产品', 'TPMS系列产品', 'ADAS标定产品']
最终相邻实体：['汽车综合诊断产品', 'TPMS系列产品', 'ADAS标定产品']
  entity='汽车综合诊断产品' -> 3 hits
  entity='TPMS系列产品' -> 3 hits
  entity='ADAS标定产品' -> 3 hits

--- seed entity: 汽车综合诊断产品 ---
[1] chunk_in_doc=76 | 现代汽车实现了高度电子化，对行业参与者要求必须有长期的技术研发和数据积累以及较强的研发创新能力，这样才能面对不断进化的汽车电子系统时开发出与之相适应的、具备全方位的诊断功能的产品，因而汽车诊断行业具有较强的行业属性和较高的技术壁垒。产品的车型覆盖面、诊断检测结果准确性、功能完整性、使用智能便利性、软...
[2] chunk_in_doc=131 | 各国对新能源以及汽车综合诊断、检测行业相关的产业政策出台将对公司产品销售产生重大影响。如果未来主要国家或地区政府对于新能源、汽车综合诊断、检测行业相关的产业政策发生重大不利变化，可能导致下游客户对公司产品的需求发生波动，进而影响公司的经营业绩。...
[3] chunk_in_doc=919 | √适用 □不适用  
单位：元币种：人民币  
| 合同分类 | 合计 |
| --- | --- |
| | 营业收入 |
| 商品类型 | |
| 汽车诊断产品 | 1,141,183,830.33 |
| TPMS 产品 | 190,832,922.58 |
| ADAS 产品 | 70,36...

--- seed entity: TPMS系列产品 ---
[1] chunk_in_doc=76 | 现代汽车实现了高度电子化，对行业参与者要求必须有长期的技术研发和数据积累以及较强的研发创新能力，这样才能面对不断进化的汽车电子系统时开发出与之相适应的、具备全方位的诊断功能的产品，因而汽车诊断行业具有较强的行业属性和较高的技术壁垒。产品的车型覆盖面、诊断检测结果准确性、功能完整性、使用智能

In [22]:
# 答案生成
from langchain_core.prompts import ChatPromptTemplate

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "基于提供的证据回答问题。1. 证据是否相关；2. 证据是否充足；3. 若证据不足请明确说明‘证据不足’。"),
    ("human", "问题：{q}\n\n证据：\n{ctx}")
])

def generate_answer(q, docs):
    ctx = "".join([d.page_content[:500] for d in docs])
    result = (ANSWER_PROMPT | llm).invoke({"q": q, "ctx": ctx})
    return result.content.strip()

print("=" * 60)
print("Naive RAG 答案")
print("=" * 60)
naive_answer = generate_answer(QUERY, naive_docs)
print(naive_answer[:600])

print("" + "=" * 60)
print("Graph RAG 答案")
print("=" * 60)
graph_answer = generate_answer(QUERY, graph_docs)
print(graph_answer[:600])

Naive RAG 答案
1. 证据不相关。提供的证据主要涉及财务报告中的费用核算、合营联营企业披露、持续经营能力声明以及报告分部的确定，并未提及“道通”公司的具体产品信息。
2. 证据不足。证据中完全没有关于“道通”公司产品的描述或列举。
3. 证据不足。
Graph RAG 答案
1. 证据相关。证据中提到了“汽车综合诊断产品”和“TPMS产品”，这些是道通的产品类别。  
2. 证据充足。根据“主营业务分产品情况”表格，明确列出了“汽车综合诊断产品”和“TPMS产品”作为分产品，因此可以确定道通有这两类产品。  
3. 无需补充。


## 总结

| 特性 | Naive RAG | Graph RAG |
|------|-----------|----------|
| 检索方式 | 向量相似度 | 向量 + 知识图谱扩展 |
| 实体感知 | ❌ | ✅ |
| 查询扩展 | 被动 | 主动（利用图关系） |
| 实现复杂度 | 低 | 中 |

**Graph RAG 的核心价值**：通过显式建模实体关系，让检索更智能——不仅能找到表面相似的内容，还能理解实体之间的关联。